# OpenPlaque v0.5 — Cross-Validation Boundary-Parameter Selection

This notebook starts from a clean Colab runtime and selects plaque-boundary refinement parameters using labeled DHM/nnU-Net sample data, not the UCLA patient case.

Expected dataset layout:

```text
Dataset001_CCTA_DHM/
  imagesTr/case001_0000.nii.gz
  labelsTr/case001.nii.gz
```

Expected model weights:

```text
/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip
```

Outputs:

```text
/content/drive/MyDrive/OpenPlaque/CV_Boundary_Tuning/
  cv_boundary_tuning_case_results.csv
  cv_boundary_tuning_summary.csv
  cv_best_boundary_parameters.json
  cv_boundary_tuning_report.html
```

Research use only. Not for clinical decision-making.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fresh runtime setup: install dependencies and put this v0.5 package on the Python path.
# Upload OpenPlaque_v0_5_CV_Boundary_Tuning.zip to /content or to /content/drive/MyDrive/OpenPlaque/.

from pathlib import Path
import os, sys, shutil, zipfile

ZIP_CANDIDATES = [
    Path('/content/OpenPlaque_v0_5_CV_Boundary_Tuning.zip'),
    Path('/content/drive/MyDrive/OpenPlaque/OpenPlaque_v0_5_CV_Boundary_Tuning.zip'),
]
EXTRACT_DIR = Path('/content/openplaque_v05_release')

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is None:
    raise FileNotFoundError('Upload OpenPlaque_v0_5_CV_Boundary_Tuning.zip to /content or /content/drive/MyDrive/OpenPlaque/.')

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(EXTRACT_DIR)

src_candidates = [EXTRACT_DIR / 'openplaque_v05' / 'src', EXTRACT_DIR / 'src']
SRC_DIR = next((p for p in src_candidates if (p / 'openplaque').exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError('Could not find src/openplaque in the v0.5 ZIP.')

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Using OpenPlaque v0.5 source:', SRC_DIR)

!pip install -q numpy pandas scipy matplotlib SimpleITK pydicom nibabel
# nnU-Net is only needed if you generate predictions in this notebook.
!pip install -q nnunetv2 acvl-utils batchgenerators batchgeneratorsv2 dynamic-network-architectures


In [ ]:
# GPU check. If this fails, switch Colab runtime to GPU before generating nnU-Net predictions.
!nvidia-smi

In [ ]:
# Configure nnU-Net folders
import os
from pathlib import Path

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'

for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('nnUNet_raw:', os.environ['nnUNet_raw'])
print('nnUNet_results:', os.environ['nnUNet_results'])

In [ ]:
# Locate or extract the labeled sample dataset.
# Put the dataset folder or dataset ZIP in one of the paths below, or edit DATASET_DIR directly.

from pathlib import Path
import zipfile, shutil

DATASET_DIR_CANDIDATES = [
    Path('/content/drive/MyDrive/OpenPlaque/Dataset001_CCTA_DHM'),
    Path('/content/nnUNet_raw/Dataset001_CCTA_DHM'),
    Path('/content/Dataset001_CCTA_DHM'),
]
DATASET_ZIP_CANDIDATES = [
    Path('/content/drive/MyDrive/OpenPlaque/Dataset001_CCTA_DHM.zip'),
    Path('/content/drive/MyDrive/OpenPlaque/Dataset001_CCTA_DHM_raw.zip'),
    Path('/content/Dataset001_CCTA_DHM.zip'),
]

DATASET_DIR = next((p for p in DATASET_DIR_CANDIDATES if (p / 'imagesTr').exists() and (p / 'labelsTr').exists()), None)
if DATASET_DIR is None:
    zpath = next((p for p in DATASET_ZIP_CANDIDATES if p.exists()), None)
    if zpath is not None:
        print('Extracting dataset ZIP:', zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall('/content/nnUNet_raw')
        DATASET_DIR = Path('/content/nnUNet_raw/Dataset001_CCTA_DHM')

if DATASET_DIR is None or not (DATASET_DIR / 'imagesTr').exists() or not (DATASET_DIR / 'labelsTr').exists():
    raise FileNotFoundError('Could not find DHM sample data. Need Dataset001_CCTA_DHM/imagesTr and labelsTr. Upload/copy the raw dataset to Drive, then rerun this cell.')

print('Using dataset:', DATASET_DIR)
!find "$DATASET_DIR" -maxdepth 2 -type f | head -20

In [ ]:
# Extract nnU-Net model weights if predictions need to be generated.
# If you already have prediction folders, this cell can still be run safely.

from pathlib import Path
import zipfile

model_zip = Path('/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip')
model_target = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')

if model_target.exists():
    print('Model already extracted:', model_target)
else:
    if model_zip.exists():
        print('Extracting model ZIP:', model_zip)
        with zipfile.ZipFile(model_zip) as z:
            z.extractall('/content/nnUNet_results')
    else:
        print('Model ZIP not found. Prediction generation will fail unless nnUNet_results is already populated.')

!find /content/nnUNet_results -maxdepth 4 | head -80

In [ ]:
# Collect labeled sample cases and create CV folds.
# Start small. Increase MAX_CASES_TOTAL after confirming the workflow.

from openplaque.cv_tuning import collect_dhm_cases, make_kfold_splits

MAX_CASES_TOTAL = 30      # set None to use all labeled cases, but this can take a long time
N_SPLITS = 5
RANDOM_SEED = 17

cases = collect_dhm_cases(DATASET_DIR, limit=MAX_CASES_TOTAL)
folds = make_kfold_splits(cases, n_splits=N_SPLITS, seed=RANDOM_SEED)

print('Cases:', len(cases))
for i, fold_cases in enumerate(folds):
    print(f'Fold {i}: {len(fold_cases)} validation cases; first:', fold_cases[0].case_id if fold_cases else None)

In [ ]:
# Generate held-out predictions for each fold.
# This is the slow/GPU step. It uses model fold 0 for validation fold 0, etc.
# If only fold_0 exists, set FOLDS_TO_RUN = [0] and N_SPLITS = 1 above.

from pathlib import Path
from openplaque.cv_tuning import run_nnunet_predictions_for_fold

PREDICTION_ROOT = Path('/content/drive/MyDrive/OpenPlaque/CV_Boundary_Tuning/predictions')
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_NAME_OR_ID = 'Dataset001_CCTA_DHM'
CONFIGURATION = '3d_fullres'
FOLDS_TO_RUN = list(range(len(folds)))
OVERWRITE_PREDICTIONS = False

prediction_dirs = {}
for fold_idx in FOLDS_TO_RUN:
    prediction_dirs[fold_idx] = run_nnunet_predictions_for_fold(
        folds[fold_idx],
        fold=fold_idx,
        output_root=PREDICTION_ROOT,
        dataset_name_or_id=DATASET_NAME_OR_ID,
        configuration=CONFIGURATION,
        overwrite=OVERWRITE_PREDICTIONS,
    )
    print('Fold', fold_idx, 'predictions:', prediction_dirs[fold_idx])

In [ ]:
# Evaluate boundary-refinement parameter grid against ground-truth labels.
# This is CPU-bound and much faster than nnU-Net prediction.

import pandas as pd
from openplaque.cv_tuning import evaluate_fold_grid, aggregate_cv_results, select_best_parameters

parameter_grid = {
    'min_component_voxels': [1, 5, 10, 25, 50],
    'lumen_distance_voxels': [0, 1, 2],
    'high_hu_threshold': [None, 850, 1000],
    'low_hu_threshold': [None, -100],
    'erode_core': [False],
    'erosion_iterations': [1],
}

fold_frames = []
for fold_idx, pred_dir in prediction_dirs.items():
    fold_frames.append(evaluate_fold_grid(
        fold=fold_idx,
        cases=folds[fold_idx],
        pred_dir=pred_dir,
        parameter_grid=parameter_grid,
    ))

cv_results = pd.concat(fold_frames, ignore_index=True)
cv_summary = aggregate_cv_results(cv_results)
best_cv_params = select_best_parameters(cv_results)

print('Case-level result rows:', len(cv_results))
print('
Best cross-validated parameters:')
print(best_cv_params)

display(cv_summary.head(20))

In [ ]:
# Save outputs to Drive
from pathlib import Path
from openplaque.cv_tuning import save_cv_outputs

OUTPUT_DIR = Path('/content/drive/MyDrive/OpenPlaque/CV_Boundary_Tuning')
paths = save_cv_outputs(cv_results, OUTPUT_DIR)

for name, path in paths.items():
    print(f'{name}: {path}')

In [ ]:
# Display best parameters later without recomputing.
from openplaque.cv_tuning import aggregate_cv_results, select_best_parameters

cv_summary = aggregate_cv_results(cv_results)
best_cv_params = select_best_parameters(cv_results)

print('Best cross-validated boundary-refinement parameters')
print('-' * 60)
for k, v in best_cv_params.items():
    print(f'{k}: {v}')

display(cv_summary.head(10))

In [ ]:
# Optional: apply the selected parameters to a UCLA/new-patient report later.
# Example after you have lad_report, rca_report, lcx_report from your patient notebook:
#
# from openplaque.boundary import refine_plaque_mask
# refinements = {}
# for report in [lad_report, rca_report, lcx_report]:
#     refinements[report.name] = refine_plaque_mask(
#         volume=report.volume,
#         mask=report.mask,
#         spacing=report.mask_image.GetSpacing(),
#         **best_cv_params,
#     )
#     refinements[report.name].summary()
